---
# Configuración
***
**DESCRIPCIÓN :**  
Este notebook realiza las siguientes funciones:  
1- Mapeo de niveles de ejecución de KPIs.  
2- Construcción de consultas.

In [0]:
%run ../Funciones

#Carga de Tablas
___

In [0]:
# Diccionario que define los esquemas y las tablas que serán cargadas (<esquema> : [<tablas>])
dict_load_tables = {
    "markpi": ["Dim_Kpi","Configuracion","Dim_Escenario"]    
}

# Carga las tablas definidas en el diccionario dentro de DataFrames para su posterior procesamiento
dfs = load_tables(dict_load_tables)

#Parametros
____

In [0]:
# Lista de identificadores de widgets
widgets = ['Escenario']

# Crear widgets del notebook de Databricks
create_widgets(widgets)

In [0]:
# Obtener los valores de los widgets y almacenarlos en el diccionario 'param'
param = get_widget_values(widgets)

In [0]:
param.update(get_dimensions_description(dfs, param))
scenario_name = param.get("ScenarioName", "default").lower()

#Configuracion
____

In [0]:
cfg = dfs["Configuracion"].alias("cfg")
kpi = dfs["Dim_Kpi"].alias("kpi")
dim_esc = dfs["Dim_Escenario"]

raw_escenario = param.get("Escenario")

if raw_escenario:
    escenarios = [int(x.strip()) for x in raw_escenario.split(",")]
else:
    escenarios = [
        int(r.id_escenario)
        for r in (
            cfg
            .filter(F.col("tipo_calculo") == "Comentario")
            .select("id_escenario")
            .distinct()
            .collect()
        )
    ]

print(f"[INFO] Escenarios a procesar: {escenarios}")

In [0]:
for escenario in escenarios:

    scenario_row = (
        dim_esc
        .filter(F.col("Id_Escenario") == escenario)
        .select("Escenario_Nombre")
        .first()
    )

    if not scenario_row:
        print(f"[WARN] Escenario {escenario} no encontrado en DimEscenario, se omite.")
        continue

    scenario_name = scenario_row.Escenario_Nombre.lower().replace(" ", "_")
    print(f"[INFO] Procesando escenario {escenario} -> {scenario_name}")

    df = (
        cfg
        .filter(
            F.col("cfg.id_kpi").isNotNull()
            & (F.col("cfg.id_escenario") == escenario)
            & (F.col("cfg.tipo_calculo") == 'Comentario')
        )
        .join(
            kpi.filter(F.col("kpi.Id_Kpi").isNotNull()),
            F.col("cfg.id_kpi") == F.col("kpi.Id_Kpi"),
            "inner"
        )
        .select(
            F.col("cfg.id_kpi"),
            F.col("cfg.id_escenario"),
            F.col("cfg.tipo_calculo"),
            F.col("cfg.fuente_datos"),
            F.col("cfg.columna_valor"),
            F.col("cfg.columna_filtro"),
            F.col("cfg.id_calculo_kpi"),
            F.col("kpi.Agregacion"),
            F.col("kpi.kpi_nombre_tecnico"),
            F.col("kpi.Id_Proceso"),
            F.col("kpi.Id_Equipo")
        )
    )

    if df.limit(1).count() == 0:
        print(f"[INFO] Escenario {escenario} sin registros, se omite.")
        continue

    valores = [
        row["fuente_datos"]
        for row in (
            df.select("fuente_datos")
            .where("fuente_datos IS NOT NULL")
            .distinct()
            .collect()
        )
    ]
    config_table_names = {x.lower(): x for x in valores if x != ""}

    table_names = (
        spark.table(f"{silver_env}.information_schema.tables")
        .filter(F.col("table_name").isin(list(config_table_names.keys())))
        .withColumn(
            "Catalogfuente_datos",
            F.concat(
                F.col("table_catalog"),
                F.lit("."),
                F.col("table_schema"),
                F.lit("."),
                F.col("table_name")
            )
        )
    )

    mapping_expr = F.create_map([F.lit(x) for kv in config_table_names.items() for x in kv])
    table_names = table_names.withColumn(
        "fuente_datos",
        mapping_expr[F.col("table_name")]
    )
    table_names = table_names.select("Catalogfuente_datos", "fuente_datos")

    df = df.join(
        table_names,
        on="fuente_datos",
        how="left"
    )

    df = df.withColumn(
        "Consulta_Diaria",
        F.concat(
            F.lit("Select DateKey, "),
            F.col("columna_valor"),
            F.lit(" AS Diario FROM "),
            F.col("Catalogfuente_datos"),
            F.lit(" where "),
            F.col("columna_filtro")
        )
    )

    df_spark = (
        df
        .withColumn("Id_Proceso", F.col("Id_Proceso").cast("int"))
        .withColumn("Id_Equipo", F.col("Id_Equipo").cast("int"))
        .withColumn("Agregacion", F.col("Agregacion").cast("string"))
    )

    table_name = f"{bronze_env}.configuracion.configuracion_{scenario_name}_Comentarios"

    df_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)

    print(f"[SUCCESS] Tabla generada: {table_name}")


In [0]:
#display(df_spark)

In [0]:
import json

escenarios_existentes = [
    int(r.id_escenario)
    for r in (
        dfs["Configuracion"]
        .filter(F.col("tipo_calculo") == "Comentario")
        .select("id_escenario")
        .distinct()
        .orderBy("id_escenario")
        .collect()
    )
]

print(f"[OUTPUT] Escenarios detectados: {escenarios_existentes}")

dbutils.notebook.exit(json.dumps(escenarios_existentes))